# Análise: Preço vs. Teor das Avaliações

**Pergunta de pesquisa:** É possível encontrar uma relação entre o preço de um jogo e o teor médio das avaliações que ele recebe dos usuários?

**Responsável:** Arthur

In [ ]:
import os, sys
from pathlib import Path

if "google.colab" in sys.modules:
    !git clone -q https://github.com/pbarruetavena/tp_ciencia_de_dados.git
    %cd tp_ciencia_de_dados
    !pip install -q kagglehub
    !python -m src.download_data
    sys.path.insert(0, os.getcwd())
else:
    # Permite importar src/ a partir de notebooks/<pasta>/
    sys.path.insert(0, str(Path.cwd().parents[1]))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns

from src.config import FIGURAS
from src.dados import carregar_jogos
from src.limpeza import com_reviews_minimas, filtrar_iqr, limpar

jogos = limpar(carregar_jogos())

# Filtrandos jogos com pelo menos 20 avaliaçoes para evitar distorções estatísticas
df_analise = com_reviews_minimas(jogos, n=20).dropna(
    subset=["price", "taxa_positiva"]
)
print(f"Tamanho da base filtrada para análise: {df_analise.shape[0]} jogos")

## Análise exploratória

**Linha de raciocínio:**
* **Expectativa inicial:** Inicialmente, poderíamos hipotetizar que jogos mais caros geram maiores expectativas nos jogadores, o que poderia resultar em avaliações mais rigorosas, e portanto, taxas positivas potencialmente menores. Em contrapartida, jogos gratuitos costumam atrair públicos muito amplos, o que pode gerar volatilidade nas avaliações. Então, é uma dúvida bem válida e legal de se explorar.
* **O que os gráficos mostram:** A visualização de dispersão mostra como a taxa de avaliações positivas (`taxa_positiva`) se distribui ao longo de diferentes faixas de preço (`price`). Além disso, o comparativo entre jogos gratiutos e pagos revela se existe uma diferença estrutural no teor das avaliações entre esses dois modelos.

In [ ]:
# Gráfico 1: dispersão entre Preço e Taxa de Avaliações Positivas (a taxa é um novo campo de análise apóos a limpeza dos dados)
# Limitamos o preço dos jogos até 60 dollares, pois a enorme maioria dos jogos nao passa desse valor, melhorando a visualização
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=df_analise[df_analise["price"] <= 60],
    x="price",
    y="taxa_positiva",
    alpha=0.3,
    color="royalblue",
    ax=ax,
)

ax.set_title("Relação entre Preço e Taxa de Avaliações Positivas")
ax.set_xlabel("Preço (US$)")
ax.set_ylabel("Taxa de Avaliações Positivas")

plt.show()

# Gráfico 2: Boxplot da Taxa Positiva por Status de Preço (gratuito vs pago)
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(
    data=df_analise, x="price_status", y="taxa_positiva", palette="Set2", ax=ax
)

ax.set_title("Distribuição da Taxa Positiva: Jogos Gratuitos vs. Pagos")
ax.set_xlabel("Status de Preço")
ax.set_ylabel("Taxa de Avaliações Positivas")

plt.show()

## Teste de hipótese e intervalo de confiança

Para verificar formalmente se existe relação estatística entre o preço e a satisfação dos usuários(medida pela taxa de avaliações positivas), adotamos duas abordagens complementares:
1. **Teste de Correlação de Spearman:** Avalia a existência de relação monotônica entre o preço contínuo e a taxa de avaliações positivas (escolhido por não assumir normalidade nas distribuições).
   * **$H_0$:** Não há correlação entre o preço do jogo e a sua taxa de avaliações positivas ($\rho = 0$).
   * **$H_1$:** Há correlação entre o preço do jogo e a sua taxa de avaliações positivas ($\rho \neq 0$).

2. **Teste de Mann-Whitney U:** Compara as medianas das taxas positivas entre jogos gratuitos e pagos.
   * **$H_0$:** A distribuição da taxa de avaliações positivas é igual entre jogos gratuitos e pagos.
   * **$H_1$:** A distribuição da taxa de avaliações positivas difere entre jogos gratuitos e pagos.

* **Nível de significância:** $\alpha = 0,05$

In [ ]:
# 1) Teste de Correlação de Spearman
preco = df_analise["price"]
taxa = df_analise["taxa_positiva"]

corr_coef, p_valor_corr = stats.spearmanr(preco, taxa)

print(f"--- Correlação de Spearman ---")
print(f"Coeficiente (rho): {corr_coef:.4f}")
print(f"P-valor: {p_valor_corr:.4e}")

#2) Teste de Mann-Whitney U (Gratuito vs pago)
gratuito_taxa = df_analise[df_analise["gratuito"] == True]["taxa_positiva"]
pago_taxa = df_analise[df_analise["gratuito"] == False]["taxa_positiva"]

stat_mann, p_valor_mann = stats.mannwhitneyu(
    gratuito_taxa, pago_taxa, alternative="two-sided"
)

print(f"\n--- Teste de Mann-Whitney U (Free vs Paid) ---")
print(f"Estatística U: {stat_mann:.2f}")
print(f"P-valor: {p_valor_mann:.4e}")

# cálculo da mediana para contexto nos resultados
print(f"\nMediana da taxa positiva (Gratuitos): {gratuito_taxa.median():.4f}")
print(f"Mediana da taxa positiva (Pagos): {pago_taxa.median():.4f}")

**Resultado:** 
* Embora os testes estatísticos tenham apresentado $p$-valores menores que o nível de significância $\alpha = 0,05$ (devido ao grande volume amostral da base de dados), o **coeficiente de correlação de Spearman foi extremamente próximo de zero ($\rho = 0,0822$)**, indicando uma força de correlação desprezível.
* O teste de Mann-Whitney U apontou diferença estatística entre jogos gratuitos e pagos, mas a diferença prática entre as medianas das taxas positivas é mínima (81,8% para pagos contra 79,1% para gratuitos, uma diferença de cerca de 2,6 pontos percentuais).

**Conclusão:** Rejeitamos formalmente $H_0$ estritamente do ponto de vista matemático devido ao tamanho da amostra, mas **concluímos que não há uma relação prática ou relevante entre o preço de um jogo e o teor médio de suas avaliações**. Os usuários avaliam jogos de forma parecida independentemente de terem custado caro, barato ou serem gratuitos.